In [ ]:
! pip install folium

In [ ]:
from astrovision.plot.plot_utils import make_mosaic
from astrovision.data import SatelliteImage, SegmentationLabeledSatelliteImage
import s3fs
import os
import geopandas as gpd
from pyproj import CRS
import matplotlib.pyplot as plt

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://" + "minio.lab.sspcloud.fr"})
out  = fs.download(rpath="projet-slums-detection/ilots", lpath="ilots", recursive=True)
out = fs.download(rpath="projet-slums-detection/data-prediction/PLEIADES/MAYOTTE", lpath="pred_mayotte", recursive=True)


In [ ]:
# Open the .gpkg file
pred_2020 = gpd.read_file('pred_mayotte/2020/predictions.gpkg')
pred_2023 = gpd.read_file('pred_mayotte/2023/predictions.gpkg')
# Print the GeoDataFrame
print(pred_2020.head())
ilots = gpd.read_file("ilots/ilots.gpkg")

#tgt_crs = CRS.from_epsg(4326)
tgt_crs = CRS.from_epsg(4471)

pred_2020 = pred_2020.to_crs(tgt_crs)
pred_2023 = pred_2023.to_crs(tgt_crs)
#pred_2020.geometry.iloc[2]

ilots = ilots.to_crs(tgt_crs)

In [ ]:

# Create a Folium map centered on the first polygon in the GeoDataFrame
center = list(pred_2020.iloc[0]['geometry'].centroid.coords)[0]
# initialisation de la carte
m = folium.Map(location=center, zoom_start=10)
# Add a GeoJSON layer from the GeoDataFrame to the map
folium.GeoJson(pred_2020).add_to(m)

ilots = ilots.to_crs(tgt_crs)

In [ ]:
pred_2020["area"] = pred_2020.area
pred_2023["area"] = pred_2023.area
pred_2020["area"]
pred_2023["area"]


# code dep : 
- 971 Guadeloupe
- 972 MArtinique
- 973 Guyane
- 974 Réunion
- 976 Mayotte

In [ ]:
ilotsMayotte = ilots[ilots["ident_ilot"].str.startswith('976')]
ilotsMayotte.plot()

In [ ]:
ilotsMayotte["area"] = ilotsMayotte.geometry.area
ilotsMayotte["area"]
print(ilotsMayotte)

## Trouver les valeurs les plus hautes (aire)

Rien de bien compliqué, mais ça sera utile pour calculer les indicateurs, les éléments extrêmes...

Par exemple, si on crée une table avec les modifications d'une année à l'autre, il suffit d'observer les îlots avec le plus de différence.

Probablement des moyens plus simples de le faire, mais j'explore...

In [ ]:
def valeurs_extremes(tableau, n, colonne, mode='top'):
    """
    Objectif de la fonction : Sélectionner et renvoyer les n lignes du tableau avec les valeurs les plus hautes ou les plus basses de la colonne spécifiée.
    
    Paramètres :
    - tableau (list): Le tableau de données à traiter. Chaque élément du tableau est supposé être un dictionnaire représentant une ligne de données.
    - n (int): Le nombre de lignes à sélectionner.
    - colonne (str): Le nom de la colonne selon laquelle les valeurs seront comparées.
    - mode (str, optionnel): Le mode de sélection, 'top' pour les valeurs les plus hautes, 'bottom' pour les valeurs les plus basses. Par défaut, 'top'.
    
    Sortie :
    list: Le tableau contenant les n lignes sélectionnées.
    """

    gdf_sorted = tableau.sort_values(by=colonne, ascending=(mode != 'top'))

    lignes = gdf_sorted.head(n)

    return lignes

In [ ]:
print(valeurs_extremes(ilotsMayotte, 5, 'area', 'top'))

In [ ]:
top_2020 = valeurs_extremes(pred_2020, 20, 'area', 'top')
top_2023 = valeurs_extremes(pred_2023, 20, 'area', 'top')

# Create a figure and axes object
fig, ax = plt.subplots()

# Plot the first GeoDataFrame in red
ilotsMayotte.plot(ax=ax, color='red', alpha=0.5)

# Plot the second GeoDataFrame in green
top_2020.plot(ax=ax, color='green', alpha=0.5)

# Plot the third GeoDataFrame in blue
top_2023.plot(ax=ax, color='blue', alpha=0.5)

# Show the plot
plt.show()

## Découpage des bâtiments pour qu'ils n'appartiennent qu'à un seul îlot

Ce découpage semblait être une bonne idée au début, pour avoir des polygones sur un seul îlot
mais l'intersection est en fait assez simple, donc peut-être pas pertinent
mais ça m'a aidé à comprendre geopandas un peu.

Le but est donc de découper les polygones, sauf qu'il y a peut-être des erreurs. On le vérifiera après.

In [ ]:
import pandas as pd

# Assurez-vous que vous avez défini vos GeoDataFrames ilots et pred_2020 correctement avant d'exécuter ce code

# Créez nouvelle_table avec la bonne colonne de géométrie
nouvelle_table = gpd.GeoDataFrame()




In [ ]:
# Corriger les géométries invalides dans ilots
ilots['geometry'] = ilots['geometry'].buffer(0)

# Corriger les géométries invalides dans pred_2020
pred_2020['geometry'] = pred_2020['geometry'].buffer(0)



In [ ]:

# Vérifiez le type des colonnes de géométrie
print(type(ilots['geometry']))
print(type(pred_2020['geometry']))


In [ ]:
ilot_test_geom.plot()

In [ ]:
ilot_test.iloc[0].geometry

In [ ]:
pred_2020
ilotsMayotte
ilot_test_geom
pred_2020.intersects(ilot_test_geom)
ilot_test_geom.crs

In [ ]:
pred_2020.intersects(ilot_test_geom)
pred_2020


In [ ]:
import pandas as pd

pred_2020 = pred_2020.assign(ident_pred=pd.Series(range(1, len(intersection)+1), index=intersection.index))
intersection = gpd.overlay(pred_2020, ilotsMayotte, how='intersection')


In [ ]:
inters.iloc[0]

In [ ]:

counts = intersection.groupby(['ident_pred']).size().reset_index(name='count')
print(counts.loc[counts["count"] > 1])

ident_pred = 26757
code_ilot1 = "976150104"
code_ilot2 = "976150108"

ilots1 = ilotsMayotte.loc[ilotsMayotte.ident_ilot == code_ilot1]
ilots2 = ilotsMayotte.loc[ilotsMayotte.ident_ilot == code_ilot2]

inters  = intersection.loc[intersection["ident_pred"]==ident_pred]
print(inters)

# Create a figure and axes object
fig, ax = plt.subplots()

# Plot the second GeoDataFrame in green
ilots1.plot(ax=ax, color='green', alpha=0.5)

# Plot the first GeoDataFrame in red
ilots2.plot(ax=ax, color='red', alpha=0.5)

inters.plot(ax=ax, color='yellow', alpha=1)

# Show the plot
plt.show()



In [ ]:
ilots1_folium = ilots1.to_crs(4326)
ilots2_folium = ilots2.to_crs(4326)
inters_folium = inters.to_crs(4326)

center = list(ilots1_folium.iloc[0]['geometry'].centroid.coords)[0]
center =  (center[1],center[0])
m = folium.Map(location=center, zoom_start=10)
folium.GeoJson(ilots1).add_to(m)
folium.GeoJson(ilots2).add_to(m)

def style_function(feature):
    return {'fillColor': 'yellow', 'color': 'black', 'weight': 3, 'fillOpacity': 0.5}

def style_function2(feature):
    return {'fillColor': 'red', 'color': 'black', 'weight': 3, 'fillOpacity': 0.5}

folium.GeoJson(inters_folium.iloc[0].geometry, style_function= style_function).add_to(m)
folium.GeoJson(inters_folium.iloc[1].geometry, style_function=style_function2).add_to(m)

m

In [ ]:
intersection

In [ ]:
intersection = gpd.overlay(pred_2020, ilotsMayotte, how='intersection')

# Create a figure and axes object
fig, ax = plt.subplots()

# Plot the second GeoDataFrame in green
ilotsMayotte.plot(ax=ax, color='green', alpha=0.5)

# Plot the first GeoDataFrame in red
intersection.plot(ax=ax, color='red', alpha=0.5)

# Show the plot
plt.show()

In [ ]:
ilot_test  =   valeurs_extremes(ilotsMayotte, 20, 'area', 'top')
ilot_test_geom =  gpd.GeoSeries(ilot_test.iloc[0].geometry).set_crs(4471)
pred_2020 = pred_2020.set_index('id') # replace 'id' with the name of the column used as index in pred_2020
ilot_test_geom.index = ilot_test['id'] # replace 'id' with the name of the column used as index in ilot_test

# Use the .intersects() method to compare the geometries
intersection = pred_2020.intersects(ilot_test_geom)
polygones_intersectes = pred_2020[pred_2020.intersects(ilot_test_geom)]

# Create a figure and axes object
fig, ax = plt.subplots()

# Plot the first GeoDataFrame in red
ilot_test_geom.plot(ax=ax, color='red', alpha=0.02)

# Plot the second GeoDataFrame in green
polygones_intersectes.plot(ax=ax, color='green', alpha=0.5)

# Show the plot
plt.show()

In [ ]:
# Parcourir chaque îlot
for i in range(len(ilots)):
    ilot = ilots.iloc[i]
    
    # Sélectionner les polygones de pred_2020 qui intersectent cet îlot
    polygones_intersectes = pred_2020[pred_2020.intersects(ilot.geometry)]
    
    # Si aucun polygone n'intersecte, passer à l'îlot suivant
    if polygones_intersectes.empty:
        continue
    
    # Effectuer l'intersection entre les polygones de pred_2020 et l'îlot
    polygones_decoupes = polygones_intersectes.intersection(ilot.geometry)
    
    # Ajouter les polygones découpés à la nouvelle table
    nouvelle_table = gpd.GeoDataFrame(pd.concat([nouvelle_table, polygones_decoupes], ignore_index=True))

In [ ]:
print(nouvelle_table)
print(pred_2020.geometry)

nouvelle_table = nouvelle_table.set_geometry(nouvelle_table[0])


In [ ]:
print(pred_2020.crs)
print(ilots.crs)
bat_decoupes = nouvelle_table
print(bat_decoupes.crs)

## Aires par îlot

On va aussi vérifier si c'est les bons résultats pour le découpage d'avant

In [ ]:
# Importer la librairie pandas
import pandas as pd

# Parcourir chaque îlot
for i in range(len(ilots)):
    ilot = ilots.iloc[i]
    
    # Sélectionner les polygones de pred_2020 qui intersectent cet îlot
    polygones_intersectes = pred_2020[pred_2020.intersects(ilot.geometry)]
    
    # Si aucun polygone n'intersecte, passer à l'îlot suivant
    if polygones_intersectes.empty:
        continue
    
    # Effectuer l'intersection entre les polygones de pred_2020 et l'îlot
    polygones_decoupes = polygones_intersectes.intersection(ilot.geometry)
    
    # Ajouter les polygones découpés à la nouvelle table
    nouvelle_table = gpd.GeoDataFrame(pd.concat([nouvelle_table, polygones_decoupes], ignore_index=True))
    
    # Calculer l'aire construite pour cet îlot
    aire_construite = polygones_decoupes.area.sum()
    
    # Ajouter l'aire construite à la table des îlots
    ilotsaire = ilots
    ilotsaire.loc[i, 'aire_construite2020'] = aire_construite

#ilotsaire = ilotsaire.dropna(subset=['aire_construite2020'])


In [ ]:
#print(ilotsaire.aire_construite2020)

print(valeurs_extremes(ilotsaire, 15, 'aire_construite2020', 'top'))

Pas trop de moyens de vérifier pour l'instant, mais ça semble à peu près pas trop mal.

## Pareil pour 2023

In [ ]:
for i in range(len(ilots)):
    ilot = ilots.iloc[i]
    
    polygones_intersectes = pred_2023[pred_2023.intersects(ilot.geometry)]

    if polygones_intersectes.empty:
        continue
    
    polygones_decoupes = polygones_intersectes.intersection(ilot.geometry)
    nouvelle_table = gpd.GeoDataFrame(pd.concat([nouvelle_table, polygones_decoupes], ignore_index=True))
    aire_construite = polygones_decoupes.area.sum()

    ilotsaire.loc[i, 'aire_construite2023'] = aire_construite

    
#ilotsaire = ilotsaire.dropna(subset=['aire_construite2023'])


In [ ]:
#print(ilotsaire.aire_construite2023)

print(valeurs_extremes(ilotsaire, 15, 'aire_construite2023', 'top'))

## On se penche sur les différences

In [ ]:
# Supprimer les lignes où les valeurs de aire_construite2020 et aire_construite2023 sont égales
ilotsairetest = ilotsaire
ilotsairetest = ilotsairetest.drop_duplicates(subset=['aire_construite2020', 'aire_construite2023'], keep=False)


In [ ]:
print(ilotsairetest)

In [ ]:
# Créer une nouvelle colonne contenant la différence entre les valeurs de aire_construite2020 et aire_construite2023
ilotsairetest['difference_aire_construite'] = ilotsairetest['aire_construite2020'] - ilotsairetest['aire_construite2023']
print(ilotsairetest)

In [ ]:
print(valeurs_extremes(ilotsairetest, 50, 'difference_aire_construite', 'top'))

In [ ]:
print(ilots)